In [14]:
from neo4j import GraphDatabase

uri = "bolt://localhost:7687"
username = "neo4j"
password = "12345678"
auth=(username, password)
driver = GraphDatabase.driver(uri, auth=(username, password))

In [15]:
import os

os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Neo4jMapReduce") \
    .master("local[*]") \
    .getOrCreate()

In [16]:
def Map(driver):

    query = """
    MATCH (c)
    WHERE c.kind = "Compound"
    OPTIONAL MATCH (c)-[r]->()
    RETURN c.name AS Compound, r.metaedge AS metaedge
    """
    with driver.session() as session:
        result = [dict(record) for record in session.run(query)]

    rdd = spark.sparkContext.parallelize(result)
    pairs = rdd.map(lambda x: (x['Compound'], x['metaedge']))
    return pairs

In [17]:
def Sort(pairs):

    gene_types = ['CbG', 'CuG', 'CdG']
    genes = pairs.filter(lambda x: x[1] in gene_types).map(lambda x: (x[0], 1))

    disease_types = ['CtD', 'CpD']
    diseases = pairs.filter(lambda x: x[1] in disease_types).map(lambda x: (x[0], 1))
    
    return genes, diseases


In [18]:
def Reduce(genes, diseases):

    gene_counts = genes.reduceByKey(lambda a, b: a + b).collect()
    disease_counts = diseases.reduceByKey(lambda a, b: a + b).collect()
    
    gene_dict = dict(gene_counts)
    disease_dict = dict(disease_counts)
    all_compounds = set(gene_dict.keys()).union(set(disease_dict.keys()))

    triples = []
    for compound in all_compounds:
        gene_count = gene_dict.get(compound, 0)
        disease_count = disease_dict.get(compound, 0)
        triples.append((compound, gene_count, disease_count))
    
    return triples
   


In [19]:
def MapReduce(driver):
    pairs = Map(driver)
    genes, diseases = Sort(pairs)
    result = Reduce(genes, diseases)

    return result

In [ ]:
result = MapReduce(driver)
sorted_data = sorted(result, key=lambda x: x[1], reverse=True)

[('Crizotinib', 585, 1),
 ('Dasatinib', 564, 1),
 ('Doxorubicin', 532, 17),
 ('Vinblastine', 523, 7),
 ('Digoxin', 522, 2)]

In [22]:
five = sorted_data[:5]
for compound, gene, disease in five:
    print(f"Compound: {compound}, Gene Count: {gene}, Disease Count: {disease}")

Compound: Crizotinib, Gene Count: 585, Disease Count: 1
Compound: Dasatinib, Gene Count: 564, Disease Count: 1
Compound: Doxorubicin, Gene Count: 532, Disease Count: 17
Compound: Vinblastine, Gene Count: 523, Disease Count: 7
Compound: Digoxin, Gene Count: 522, Disease Count: 2


In [21]:
spark.stop()
driver.close()